In [1]:
# Verify the actual accelerator before benchmarking.
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    raise RuntimeError('A CUDA GPU runtime is required for this benchmark.')

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
VRAM GiB: 14.56


In [2]:
# Install the same dependencies for both pinned Klygo revisions.
import os, sys, subprocess, tempfile
from pathlib import Path
ROOT = Path(tempfile.mkdtemp(prefix='klygo-gpu-', dir='/content'))
NEW = ROOT / 'optimized'
OLD = ROOT / 'baseline'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir()
BASE_SHA = 'fb0f42f4075cb2bfab250446c1df059a67b5b2bb'
NEW_SHA = '9ff1abd3f1b39f594144d9a81373e7109998bf86'
def command(args):
    result = subprocess.run([str(x) for x in args], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(result.stdout[-6000:])
    return result.stdout
print(command(['git', 'clone', '--quiet', '--branch', 'perf/managed-input-pipeline', '--single-branch', 'https://github.com/IchigoMazone/klygo.git', NEW]))
command(['git', '-C', NEW, 'checkout', '--quiet', '--detach', NEW_SHA])
command(['git', '-C', NEW, 'worktree', 'add', '--detach', OLD, BASE_SHA])
print('Installing dependencies...', flush=True)
print(command([sys.executable, '-m', 'pip', 'install', '-q', '-e', NEW, 'ultralytics==8.4.208']))
import cv2, numpy as np, torch, ultralytics
assert torch.cuda.is_available(), 'CUDA GPU required'
WEIGHTS = ARTIFACTS / 'yolov8n.pt'
from ultralytics import YOLO
YOLO(str(WEIGHTS))
VIDEO = ARTIFACTS / 'bus_motion_h264.mp4'
asset = Path(ultralytics.__file__).parent / 'assets' / 'bus.jpg'
image = cv2.resize(cv2.imread(str(asset)), (1280, 720))
assert image is not None
encoder = subprocess.Popen(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', '1280x720', '-r', '30', '-i', '-', '-an', '-c:v', 'libx264', '-preset', 'veryfast', '-g', '60', '-bf', '2', '-pix_fmt', 'yuv420p', str(VIDEO)], stdin=subprocess.PIPE)
for index in range(240):
    frame = np.roll(image, int(20 * np.sin(index / 15)), axis=1).copy()
    cv2.putText(frame, str(index), (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    encoder.stdin.write(frame.tobytes())
encoder.stdin.close()
assert encoder.wait() == 0
print('GPU:', torch.cuda.get_device_name(0))
print('Versions:', torch.__version__, ultralytics.__version__, cv2.__version__)
print('Video: 240 frames, 1280x720, H.264, moving public bus sample (controlled workload).')
print('Baseline:', BASE_SHA)
print('Optimized:', NEW_SHA)
print('Artifacts:', ARTIFACTS)


Installing dependencies...

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4
Versions: 2.14.1+cu130 8.4.208 4.14.0
Video: 240 frames, 1280x720, H.264, moving public bus sample (controlled workload).
Baseline: fb0f42f4075cb2bfab250446c1df059a67b5b2bb
Optimized: 9ff1abd3f1b39f594144d9a81373e7109998bf86
Artifacts: /content/klygo-gpu-ggp5tvm9/artifacts


In [3]:
# Controlled CUDA benchmark: isolated processes, fixed settings, identical input.
import json, textwrap, time
WORKER = r'''
import gc, hashlib, json, os, statistics, sys, time
from pathlib import Path
import cv2, numpy as np, torch
from ultralytics import YOLO
from klygo import files, models
cfg = json.loads(sys.argv[1])
cv2.setNumThreads(1)
torch.set_num_threads(2)
torch.manual_seed(0)
assert torch.cuda.is_available()
torch.cuda.set_device(0)
torch.backends.cudnn.benchmark = True
options = dict(device='cuda:0', imgsz=640, half=False, rect=True, conf=0.25, iou=0.7, verbose=False)
if cfg['kind'] == 'raw':
    detector = YOLO(cfg['weights'])
    def predict():
        return detector.predict(cfg['video'], batch=cfg['batch'], vid_stride=cfg['stride'], **options)
else:
    # Both revisions use the same public configuration API and local model_id.
    # This avoids the baseline's unrelated loading-only weights forwarding bug.
    config_path = Path(cfg['out']).with_suffix('.model.json')
    models.export('yolov8n.pt', config_path, overwrite=True, device='cuda:0', imgsz=640,
                  half=False, model_rect=True, threshold=0.25, iou=0.7)
    metadata = files.load(config_path, verbose=False)
    metadata['model_id'] = cfg['weights']
    metadata['model'].pop('weights', None)
    detector = models.load(metadata)
    def predict():
        if cfg['kind'] == 'native':
            return detector.predict(cfg['video'], native=True, verbose=False)
        return detector.predict(cfg['video'], batch=cfg['batch'], workers=cfg['workers'],
                                prefetch=cfg['prefetch'], vid_stride=cfg['stride'], verbose=False)
def signature(result):
    digest = hashlib.sha256()
    count = objects = 0
    for frame in result:
        if cfg['kind'] == 'raw':
            coords = frame.boxes.xyxy.detach().cpu().numpy().astype(np.float64)
            scores = frame.boxes.conf.detach().cpu().numpy().astype(np.float64)
            ids = frame.boxes.cls.detach().cpu().numpy().astype(np.int32)
        else:
            coords = np.asarray(frame.boxes, dtype=np.float64)
            scores = np.asarray(frame.scores, dtype=np.float64)
            ids = np.asarray(frame.class_ids, dtype=np.int32)
        digest.update(np.round(coords, 3).tobytes())
        digest.update(np.round(scores, 5).tobytes())
        digest.update(ids.tobytes())
        objects += len(ids)
        count += 1
    return count, objects, digest.hexdigest()
# Full-clip warmup initializes the predictor, exact input shapes, and CUDA kernels.
warm = predict()
torch.cuda.synchronize()
del warm
gc.collect()
torch.set_num_threads(2)
seconds, signatures, counts, objects, peaks = [], [], [], [], []
for repeat in range(3):
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    result = predict()
    torch.cuda.synchronize()
    seconds.append(time.perf_counter() - started)
    peaks.append(torch.cuda.max_memory_allocated() / 2**20)
    # Parity checking is outside the timed interval for all modes.
    count, object_count, digest = signature(result)
    counts.append(count)
    objects.append(object_count)
    signatures.append(digest)
    del result
data = dict(name=cfg['name'], revision=cfg['revision'], kind=cfg['kind'], batch=cfg['batch'],
            workers=cfg['workers'], prefetch=cfg['prefetch'], stride=cfg['stride'],
            fps=counts[0] / statistics.median(seconds), seconds=seconds, frames=counts,
            objects=objects, signatures=signatures, stable=len(set(signatures)) == 1,
            peak_vram_mib=max(peaks), gpu=torch.cuda.get_device_name(0),
            torch=torch.__version__, torch_threads=torch.get_num_threads())
Path(cfg['out']).write_text(json.dumps(data, indent=2))
print('RESULT', json.dumps(data), flush=True)
'''
worker_path = ARTIFACTS / 'gpu_worker.py'
worker_path.write_text(textwrap.dedent(WORKER))
CASES = []
def add(name, revision, kind, batch=1, workers=1, prefetch=1, stride=1):
    CASES.append(dict(name=name, revision=revision, kind=kind, batch=batch, workers=workers,
                      prefetch=prefetch, stride=stride, weights=str(WEIGHTS), video=str(VIDEO),
                      out=str(ARTIFACTS / (name + '.json'))))
add('raw_b1', 'raw', 'raw')
for rev in ('baseline', 'optimized'):
    add(rev + '_native_b1', rev, 'native')
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b1_w1', rev, 'managed')
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b1_w4', rev, 'managed', workers=4, prefetch=2)
add('raw_b4', 'raw', 'raw', batch=4)
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b4_w4', rev, 'managed', batch=4, workers=4, prefetch=2)
add('raw_b4_stride3', 'raw', 'raw', batch=4, stride=3)
for rev in ('baseline', 'optimized'):
    add(rev + '_managed_b4_w4_stride3', rev, 'managed', batch=4, workers=4, prefetch=2, stride=3)
REPORTS = []
print('GPU:', torch.cuda.get_device_name(0))
print('All modes: imgsz=640, FP32, conf=0.25, IoU=0.7, rect=True; one full-clip warmup + 3 measured runs.')
print('Native is compared only at batch=1. Input: controlled H.264 bus-motion clip, not the original user video.')
for number, cfg in enumerate(CASES, 1):
    checkout = OLD if cfg['revision'] == 'baseline' else NEW
    env = os.environ.copy()
    env['PYTHONPATH'] = str(checkout)
    env['YOLO_VERBOSE'] = 'False'
    print(f"[{number}/{len(CASES)}] Running {cfg['name']} ...", flush=True)
    process = subprocess.run([sys.executable, str(worker_path), json.dumps(cfg)], cwd=ROOT,
                             env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             timeout=600)
    (ARTIFACTS / (cfg['name'] + '.log')).write_text(process.stdout)
    if process.returncode:
        print(process.stdout[-8000:])
        raise RuntimeError('Benchmark failed: ' + cfg['name'])
    result = json.loads(Path(cfg['out']).read_text())
    REPORTS.append(result)
    (ARTIFACTS / 'gpu_report.json').write_text(json.dumps(REPORTS, indent=2))
    print(f"  {result['fps']:.2f} FPS | frames={result['frames'][0]} | objects={result['objects'][0]} | stable={result['stable']}", flush=True)
import pandas as pd
summary = pd.DataFrame([{k: row[k] for k in ('name', 'fps', 'batch', 'workers', 'stride', 'stable', 'peak_vram_mib')} for row in REPORTS])
display(summary.round(2))
lookup = {row['name']: row for row in REPORTS}
for suffix in ('native_b1', 'managed_b1_w1', 'managed_b1_w4', 'managed_b4_w4', 'managed_b4_w4_stride3'):
    old, new = lookup['baseline_' + suffix], lookup['optimized_' + suffix]
    same = old['signatures'][0] == new['signatures'][0]
    print(f"{suffix}: {old['fps']:.2f} -> {new['fps']:.2f} FPS ({new['fps']/old['fps']:.2f}x), rounded output parity={same}")
print('Full report:', ARTIFACTS / 'gpu_report.json')


GPU: Tesla T4
All modes: imgsz=640, FP32, conf=0.25, IoU=0.7, rect=True; one full-clip warmup + 3 measured runs.
Native is compared only at batch=1. Input: controlled H.264 bus-motion clip, not the original user video.
[1/13] Running raw_b1 ...
  85.86 FPS | frames=240 | objects=1321 | stable=True
[2/13] Running baseline_native_b1 ...
  84.69 FPS | frames=240 | objects=1321 | stable=True
[3/13] Running optimized_native_b1 ...
  78.09 FPS | frames=240 | objects=1321 | stable=True
[4/13] Running baseline_managed_b1_w1 ...
  75.13 FPS | frames=240 | objects=1321 | stable=True
[5/13] Running optimized_managed_b1_w1 ...
  64.05 FPS | frames=240 | objects=1321 | stable=True
[6/13] Running baseline_managed_b1_w4 ...
  76.14 FPS | frames=240 | objects=1321 | stable=True
[7/13] Running optimized_managed_b1_w4 ...
  64.57 FPS | frames=240 | objects=1321 | stable=True
[8/13] Running raw_b4 ...
  159.68 FPS | frames=240 | objects=1321 | stable=True
[9/13] Running baseline_managed_b4_w4 ...
  27.48

,name,fps,batch,workers,stride,stable,peak_vram_mib
0,raw_b1,85.86,1,1,1,True,34.37
1,baseline_native_b1,84.69,1,1,1,True,34.37
2,optimized_native_b1,78.09,1,1,1,True,34.37
3,baseline_managed_b1_w1,75.13,1,1,1,True,24.66
4,optimized_managed_b1_w1,64.05,1,1,1,True,24.66
5,baseline_managed_b1_w4,76.14,1,4,1,True,24.66
6,optimized_managed_b1_w4,64.57,1,4,1,True,24.66
7,raw_b4,159.68,4,1,1,True,91.39
8,baseline_managed_b4_w4,27.48,4,4,1,True,63.09
9,optimized_managed_b4_w4,122.30,4,4,1,True,63.09


native_b1: 84.69 -> 78.09 FPS (0.92x), rounded output parity=False
managed_b1_w1: 75.13 -> 64.05 FPS (0.85x), rounded output parity=False
managed_b1_w4: 76.14 -> 64.57 FPS (0.85x), rounded output parity=True
managed_b4_w4: 27.48 -> 122.30 FPS (4.45x), rounded output parity=True
managed_b4_w4_stride3: 7.91 -> 84.64 FPS (10.71x), rounded output parity=False
Full report: /content/klygo-gpu-ggp5tvm9/artifacts/gpu_report.json


In [7]:
# Persist machine-readable measurements in notebook output and show the comparison.
from IPython.display import Markdown, JSON, display
reports = json.loads((ARTIFACTS / 'gpu_report.json').read_text())
assert len(reports) == len(CASES), f'Only {len(reports)}/{len(CASES)} cases finished; inspect the previous cell.'
lookup = {r['name']: r for r in reports}
print('Pinned source verification:')
for checkout in (OLD, NEW):
    print(checkout.name, command(['git', '-C', checkout, 'rev-parse', 'HEAD']).strip())
    env = os.environ.copy()
    env['PYTHONPATH'] = str(checkout)
    print(subprocess.check_output([sys.executable, '-c', 'import klygo; print(klygo.__file__)'], env=env, cwd=ROOT, text=True))
display(JSON({'gpu': torch.cuda.get_device_name(0), 'video': str(VIDEO), 'frames': 240,
              'imgsz': 640, 'half': False, 'warmup': 'one full clip', 'repetitions': 3,
              'baseline_sha': BASE_SHA, 'optimized_sha': NEW_SHA, 'measurements': reports}, expanded=False))
lines = ['## Tesla T4: Klygo before / after',
         'Controlled H.264 clip: 240 frames, 1280x720; YOLOv8n, imgsz=640, FP32. Median of 3 runs after warmup.',
         '', '| Configuration | Raw FPS | Before FPS | After FPS | Change | Output parity |',
         '|---|---:|---:|---:|---:|---|']
for title, suffix, raw_name in [
    ('Native, batch 1', 'native_b1', 'raw_b1'),
    ('Managed, batch 1, workers 1', 'managed_b1_w1', 'raw_b1'),
    ('Managed, batch 1, workers 4', 'managed_b1_w4', 'raw_b1'),
    ('Managed, batch 4, workers 4', 'managed_b4_w4', 'raw_b4'),
    ('Managed, batch 4, workers 4, stride 3', 'managed_b4_w4_stride3', 'raw_b4_stride3')]:
    old, new, raw = lookup['baseline_' + suffix], lookup['optimized_' + suffix], lookup[raw_name]
    same = old['signatures'][0] == new['signatures'][0]
    lines.append(f"| {title} | {raw['fps']:.2f} | {old['fps']:.2f} | {new['fps']:.2f} | {(new['fps']/old['fps']-1)*100:+.1f}% | {'MATCH' if same else 'DIFF'} |")
lines += ['', '**Scope:** this is a controlled moving-image video, not the original video behind the 90/87/30 FPS observation. Raw stride-3 and managed stride-3 can select different starting frames; only before/after parity is asserted in that row.',
          '', 'All existing public API signatures were preserved. No results are inferred from CPU measurements.']
summary_md = chr(10).join(lines)
(ARTIFACTS / 'gpu_summary.md').write_text(summary_md)
display(Markdown(summary_md))


Pinned source verification:
baseline fb0f42f4075cb2bfab250446c1df059a67b5b2bb
/content/klygo-gpu-ggp5tvm9/baseline/klygo/__init__.py

optimized 9ff1abd3f1b39f594144d9a81373e7109998bf86
/content/klygo-gpu-ggp5tvm9/optimized/klygo/__init__.py



<IPython.core.display.JSON object>

## Tesla T4: Klygo before / after
Controlled H.264 clip: 240 frames, 1280x720; YOLOv8n, imgsz=640, FP32. Median of 3 runs after warmup.

| Configuration | Raw FPS | Before FPS | After FPS | Change | Output parity |
|---|---:|---:|---:|---:|---|
| Native, batch 1 | 85.86 | 84.69 | 78.09 | -7.8% | DIFF |
| Managed, batch 1, workers 1 | 85.86 | 75.13 | 64.05 | -14.8% | DIFF |
| Managed, batch 1, workers 4 | 85.86 | 76.14 | 64.57 | -15.2% | MATCH |
| Managed, batch 4, workers 4 | 159.68 | 27.48 | 122.30 | +345.1% | MATCH |
| Managed, batch 4, workers 4, stride 3 | 97.34 | 7.91 | 84.64 | +970.5% | DIFF |

**Scope:** this is a controlled moving-image video, not the original video behind the 90/87/30 FPS observation. Raw stride-3 and managed stride-3 can select different starting frames; only before/after parity is asserted in that row.

All existing public API signatures were preserved. No results are inferred from CPU measurements.

In [6]:
# Confirm batch-1 timing in reverse order and inspect numeric output differences.
import statistics
capture_worker = WORKER.replace('count = objects = 0', 'count = objects = 0\n    captured = []')
capture_worker = capture_worker.replace('objects += len(ids)', 'captured.append((coords, scores, ids))\n        objects += len(ids)')
capture_worker = capture_worker.replace('return count, objects, digest.hexdigest()', "np.savez(cfg['capture'], lengths=np.asarray([len(x[1]) for x in captured]), boxes=np.concatenate([x[0] for x in captured]), scores=np.concatenate([x[1] for x in captured]), class_ids=np.concatenate([x[2] for x in captured]))\n    return count, objects, digest.hexdigest()")
confirm_path = ARTIFACTS / 'gpu_capture_worker.py'
CONFIRM = []
confirm_names = ['optimized_native_b1', 'baseline_native_b1', 'optimized_managed_b1_w1',
                 'baseline_managed_b1_w1', 'raw_b1',
                 'baseline_managed_b4_w4_stride3', 'optimized_managed_b4_w4_stride3']
for base_name in confirm_names:
    cfg = dict(next(item for item in CASES if item['name'] == base_name))
    cfg['name'] = base_name + '_confirm'
    cfg['out'] = str(ARTIFACTS / (cfg['name'] + '.json'))
    cfg['capture'] = str(ARTIFACTS / (base_name + '.npz'))
    code = capture_worker.replace('range(3)', 'range(1)') if cfg['stride'] == 3 else capture_worker
    confirm_path.write_text(textwrap.dedent(code))
    env = os.environ.copy()
    env['PYTHONPATH'] = str(OLD if cfg['revision'] == 'baseline' else NEW)
    env['YOLO_VERBOSE'] = 'False'
    print('Confirming', base_name, flush=True)
    process = subprocess.run([sys.executable, str(confirm_path), json.dumps(cfg)], cwd=ROOT,
                             env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600)
    (ARTIFACTS / (cfg['name'] + '.log')).write_text(process.stdout)
    if process.returncode:
        print(process.stdout[-6000:])
        raise RuntimeError('Confirmation failed: ' + base_name)
    data = json.loads(Path(cfg['out']).read_text())
    CONFIRM.append(data)
    print(f"  {data['fps']:.2f} FPS", flush=True)
    (ARTIFACTS / 'gpu_confirmation.json').write_text(json.dumps(CONFIRM, indent=2))
NUMERIC_PARITY = []
for suffix in ('native_b1', 'managed_b1_w1', 'managed_b4_w4_stride3'):
    left = np.load(ARTIFACTS / ('baseline_' + suffix + '.npz'))
    right = np.load(ARTIFACTS / ('optimized_' + suffix + '.npz'))
    same_lengths = np.array_equal(left['lengths'], right['lengths'])
    same_classes = np.array_equal(left['class_ids'], right['class_ids'])
    if same_lengths and same_classes:
        box_error = float(np.max(np.abs(left['boxes'] - right['boxes']), initial=0))
        score_error = float(np.max(np.abs(left['scores'] - right['scores']), initial=0))
        within_tolerance = box_error <= 0.01 and score_error <= 1e-5
    else:
        box_error = score_error = None
        within_tolerance = False
    record = dict(configuration=suffix, same_frame_counts=same_lengths, same_class_ids=same_classes,
                  max_box_error_px=box_error, max_score_error=score_error, within_tolerance=within_tolerance)
    NUMERIC_PARITY.append(record)
    print(record)
(ARTIFACTS / 'gpu_numeric_parity.json').write_text(json.dumps(NUMERIC_PARITY, indent=2))


Confirming optimized_native_b1
  81.27 FPS
Confirming baseline_native_b1
  73.85 FPS
Confirming optimized_managed_b1_w1
  70.76 FPS
Confirming baseline_managed_b1_w1
  66.98 FPS
Confirming raw_b1
  77.86 FPS
Confirming baseline_managed_b4_w4_stride3
  9.21 FPS
Confirming optimized_managed_b4_w4_stride3
  86.79 FPS
{'configuration': 'native_b1', 'same_frame_counts': True, 'same_class_ids': True, 'max_box_error_px': 0.0, 'max_score_error': 0.0, 'within_tolerance': True}
{'configuration': 'managed_b1_w1', 'same_frame_counts': True, 'same_class_ids': True, 'max_box_error_px': 0.00048828125, 'max_score_error': 1.3709068298339844e-06, 'within_tolerance': True}
{'configuration': 'managed_b4_w4_stride3', 'same_frame_counts': True, 'same_class_ids': True, 'max_box_error_px': 0.0, 'max_score_error': 0.0, 'within_tolerance': True}


611

In [8]:
# Final reproducible report, including the confirmation runs and numeric checks.
import hashlib, zipfile
from IPython.display import Markdown, display
primary = {r['name']: r for r in REPORTS}
confirm = {r['name'].removesuffix('_confirm'): r for r in CONFIRM}
parity = {r['configuration']: r for r in NUMERIC_PARITY}
final_lines = ['## GPU benchmark results - Tesla T4',
               'YOLOv8n, imgsz=640, FP32; controlled 240-frame 1280x720 H.264 video. Each main result is the median of 3 measured runs after full-clip warmup.',
               '', '| Mode | Raw FPS | Before FPS | After FPS | Repeat before / after | Output check |',
               '|---|---:|---:|---:|---|---|']
for title, suffix, raw_name in [
    ('Native, batch 1', 'native_b1', 'raw_b1'),
    ('Managed, batch 1, workers 1', 'managed_b1_w1', 'raw_b1'),
    ('Managed, batch 1, workers 4', 'managed_b1_w4', 'raw_b1'),
    ('Managed, batch 4, workers 4', 'managed_b4_w4', 'raw_b4'),
    ('Managed, batch 4, workers 4, stride 3', 'managed_b4_w4_stride3', 'raw_b4_stride3')]:
    before, after = primary['baseline_' + suffix], primary['optimized_' + suffix]
    strict = before['signatures'][0] == after['signatures'][0]
    numeric = parity.get(suffix)
    check = 'Exact rounded match' if strict else ('Within tolerance' if numeric and numeric['within_tolerance'] else 'Needs investigation')
    repeat = '--'
    if 'baseline_' + suffix in confirm:
        repeat = f"{confirm['baseline_' + suffix]['fps']:.2f} / {confirm['optimized_' + suffix]['fps']:.2f}"
    final_lines.append(f"| {title} | {primary[raw_name]['fps']:.2f} | {before['fps']:.2f} | {after['fps']:.2f} | {repeat} | {check} |")
final_lines += ['', '**Interpretation:** batch 4 with four workers improved strongly on this workload. Batch-1 results vary between runs and do not establish a stable improvement. Do not compare different batch sizes as if they were identical configurations.',
                '', 'Numeric tolerance: identical per-frame object counts and class-ID order; maximum coordinate error <= 0.01 pixel and maximum score error <= 0.00001. Strict hash differences alone can arise from floating-point differences; numeric checks are recorded below.',
                '', 'The video was generated from the public Ultralytics bus sample with small motion. It is not the original video behind the earlier 90/87/30 FPS observation. Raw stride-3 and managed stride-3 may start at different frames; compare the two Klygo revisions within that row.',
                '', '### Numeric output checks']
for record in NUMERIC_PARITY:
    final_lines.append('- ' + str(record))
final_lines += ['', f"Raw batch-1 control: {primary['raw_b1']['fps']:.2f} FPS initially; {confirm['raw_b1']['fps']:.2f} FPS in the confirmation run.",
                '', 'Baseline commit: ' + BASE_SHA, 'Optimized commit: ' + NEW_SHA]
FINAL_MD = chr(10).join(final_lines)
(ARTIFACTS / 'gpu_final_report.md').write_text(FINAL_MD)
manifest = dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__, ultralytics=ultralytics.__version__,
                opencv=cv2.__version__, baseline=BASE_SHA, optimized=NEW_SHA, video_frames=240,
                video_sha256=hashlib.sha256(VIDEO.read_bytes()).hexdigest(),
                weights_sha256=hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(),
                primary=REPORTS, confirmation=CONFIRM, numeric_parity=NUMERIC_PARITY)
(ARTIFACTS / 'gpu_complete_report.json').write_text(json.dumps(manifest, indent=2))
BUNDLE = ARTIFACTS / 'klygo_t4_benchmark_results.zip'
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(ARTIFACTS.iterdir()):
        if path.suffix in ('.json', '.md', '.npz', '.py'):
            bundle.write(path, path.name)
display(Markdown(FINAL_MD))
print('Downloadable report:', BUNDLE)


## GPU benchmark results - Tesla T4
YOLOv8n, imgsz=640, FP32; controlled 240-frame 1280x720 H.264 video. Each main result is the median of 3 measured runs after full-clip warmup.

| Mode | Raw FPS | Before FPS | After FPS | Repeat before / after | Output check |
|---|---:|---:|---:|---|---|
| Native, batch 1 | 85.86 | 84.69 | 78.09 | 73.85 / 81.27 | Within tolerance |
| Managed, batch 1, workers 1 | 85.86 | 75.13 | 64.05 | 66.98 / 70.76 | Within tolerance |
| Managed, batch 1, workers 4 | 85.86 | 76.14 | 64.57 | -- | Exact rounded match |
| Managed, batch 4, workers 4 | 159.68 | 27.48 | 122.30 | -- | Exact rounded match |
| Managed, batch 4, workers 4, stride 3 | 97.34 | 7.91 | 84.64 | 9.21 / 86.79 | Within tolerance |

**Interpretation:** batch 4 with four workers improved strongly on this workload. Batch-1 results vary between runs and do not establish a stable improvement. Do not compare different batch sizes as if they were identical configurations.

Numeric tolerance: identical per-frame object counts and class-ID order; maximum coordinate error <= 0.01 pixel and maximum score error <= 0.00001. Strict hash differences alone can arise from floating-point differences; numeric checks are recorded below.

The video was generated from the public Ultralytics bus sample with small motion. It is not the original video behind the earlier 90/87/30 FPS observation. Raw stride-3 and managed stride-3 may start at different frames; compare the two Klygo revisions within that row.

### Numeric output checks
- {'configuration': 'native_b1', 'same_frame_counts': True, 'same_class_ids': True, 'max_box_error_px': 0.0, 'max_score_error': 0.0, 'within_tolerance': True}
- {'configuration': 'managed_b1_w1', 'same_frame_counts': True, 'same_class_ids': True, 'max_box_error_px': 0.00048828125, 'max_score_error': 1.3709068298339844e-06, 'within_tolerance': True}
- {'configuration': 'managed_b4_w4_stride3', 'same_frame_counts': True, 'same_class_ids': True, 'max_box_error_px': 0.0, 'max_score_error': 0.0, 'within_tolerance': True}

Raw batch-1 control: 85.86 FPS initially; 77.86 FPS in the confirmation run.

Baseline commit: fb0f42f4075cb2bfab250446c1df059a67b5b2bb
Optimized commit: 9ff1abd3f1b39f594144d9a81373e7109998bf86

Downloadable report: /content/klygo-gpu-ggp5tvm9/artifacts/klygo_t4_benchmark_results.zip


In [9]:
# Download the complete measured report and reproducibility files.
from google.colab import files as colab_files
colab_files.download(str(BUNDLE))

In [10]:
# Validate the batch-1 follow-up without changing either pinned checkout.
FOLLOW = ROOT / 'followup'
if not FOLLOW.exists():
    command(['git', '-C', NEW, 'worktree', 'add', '--detach', FOLLOW, NEW_SHA])
source_path = FOLLOW / 'klygo/models/detection/base.py'
original_block = r'''        for part in batches:
            yield part, [materialize(image) for image in part]
        return

    stopped = Event()
'''
replacement_block = r'''        try:
            for part in batches:
                yield part, [materialize(image) for image in part]
        finally:
            close = getattr(iterator, "close", None)
            if callable(close):
                close()
        return

    source_prepares_pixels = (
        bool(getattr(images, "_predicates", ()))
        or bool(getattr(images, "_preprocessing_ops", ()))
        or getattr(images, "_motion_threshold", None) is not None
        or getattr(images, "_deduplicate_threshold", None) is not None
    )
    if batch == 1 and workers == 1 and not source_prepares_pixels:
        # One ordered worker already serializes video reads. Avoid a second
        # executor and a completed intermediate Future on every default frame.
        executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="klygo-decode")
        pending_single: deque[tuple[Any, Future[Any]]] = deque()
        exhausted = object()

        def submit_one() -> bool:
            image = next(iterator, exhausted)
            if image is exhausted:
                return False
            pending_single.append((image, executor.submit(materialize, image)))
            return True

        try:
            for _ in range(prefetch + 1):
                if not submit_one():
                    break
            while pending_single:
                image, future = pending_single.popleft()
                yield [image], [future.result()]
                submit_one()
        finally:
            for _image, future in pending_single:
                future.cancel()
            executor.shutdown(wait=True, cancel_futures=True)
            close = getattr(iterator, "close", None)
            if callable(close):
                close()
        return

    stopped = Event()
'''
source = source_path.read_text()
if original_block in source:
    source_path.write_text(source.replace(original_block, replacement_block))
else:
    assert replacement_block in source, 'Unexpected source state'
patch = command(['git', '-C', FOLLOW, 'diff', '--', 'klygo/models/detection/base.py'])
(ARTIFACTS / 'followup_pipeline.diff').write_text(patch)
print('Follow-up source SHA256:', hashlib.sha256(source_path.read_bytes()).hexdigest(), flush=True)
FOLLOWUP_REPORTS = []
followup_path = ARTIFACTS / 'gpu_followup_worker.py'
followup_path.write_text(textwrap.dedent(capture_worker))
jobs = [('followup_b1_a', 'followup', 'optimized_managed_b1_w1'),
        ('previous_b1_a', 'previous', 'optimized_managed_b1_w1'),
        ('previous_b1_b', 'previous', 'optimized_managed_b1_w1'),
        ('followup_b1_b', 'followup', 'optimized_managed_b1_w1'),
        ('followup_b4_w4', 'followup', 'optimized_managed_b4_w4')]
for name, revision, template in jobs:
    cfg = dict(next(item for item in CASES if item['name'] == template))
    cfg.update(name=name, revision=revision, out=str(ARTIFACTS / (name + '.json')),
               capture=str(ARTIFACTS / (name + '.npz')))
    checkout = FOLLOW if revision == 'followup' else NEW
    env = os.environ.copy()
    env['PYTHONPATH'] = str(checkout)
    env['YOLO_VERBOSE'] = 'False'
    print('Running', name, flush=True)
    process = subprocess.run([sys.executable, str(followup_path), json.dumps(cfg)], cwd=ROOT,
                             env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=600)
    (ARTIFACTS / (name + '.log')).write_text(process.stdout)
    if process.returncode:
        print(process.stdout[-6000:])
        raise RuntimeError('Follow-up benchmark failed: ' + name)
    result = json.loads(Path(cfg['out']).read_text())
    result['source_sha256'] = hashlib.sha256((checkout / 'klygo/models/detection/base.py').read_bytes()).hexdigest()
    result['source_ref'] = NEW_SHA + (' plus followup_pipeline.diff' if revision == 'followup' else '')
    FOLLOWUP_REPORTS.append(result)
    (ARTIFACTS / 'gpu_followup.json').write_text(json.dumps(FOLLOWUP_REPORTS, indent=2))
    print(f"  {result['fps']:.2f} FPS", flush=True)
previous_times = [t for r in FOLLOWUP_REPORTS if r['revision'] == 'previous' for t in r['seconds']]
followup_times = [t for r in FOLLOWUP_REPORTS if r['revision'] == 'followup' and r['batch'] == 1 for t in r['seconds']]
print('Pooled batch-1 median:', 240 / statistics.median(previous_times), '->', 240 / statistics.median(followup_times))
left, right = np.load(ARTIFACTS / 'previous_b1_b.npz'), np.load(ARTIFACTS / 'followup_b1_b.npz')
FOLLOWUP_PARITY = dict(same_frame_counts=np.array_equal(left['lengths'], right['lengths']),
                      same_class_ids=np.array_equal(left['class_ids'], right['class_ids']),
                      max_box_error_px=float(np.max(np.abs(left['boxes']-right['boxes']), initial=0)),
                      max_score_error=float(np.max(np.abs(left['scores']-right['scores']), initial=0)))
print('Follow-up parity:', FOLLOWUP_PARITY)


Follow-up source SHA256: 5ac4d0bc083821a384d9517a6e2438d8f9ed4a7bee97c634905e29954589241e
Running followup_b1_a
  77.03 FPS
Running previous_b1_a
  79.03 FPS
Running previous_b1_b
  72.35 FPS
Running followup_b1_b
  77.97 FPS
Running followup_b4_w4
  128.67 FPS
Pooled batch-1 median: 75.54568461090905 -> 77.34311744093024
Follow-up parity: {'same_frame_counts': True, 'same_class_ids': True, 'max_box_error_px': 0.00048828125, 'max_score_error': 1.1920928955078125e-06}


In [11]:
# Export actual measurements; no model weights, private data, or browser state.
benchmark_export = dict(manifest)
benchmark_export['protocol'] = dict(model='YOLOv8n', imgsz=640, half=False, rect=True, conf=0.25, iou=0.7,
                                  device='cuda:0', warmup_full_clips=1, measured_repetitions=3,
                                  opencv_threads=1, torch_threads=2, cudnn_benchmark=True,
                                  timing='CUDA-synchronized prediction only; parity checks outside timing',
                                  workload='240-frame 1280x720 H.264 moving public bus sample, GOP 60, B-frames 2',
                                  native_batch=1, raw_stride3_start_may_differ=True)
benchmark_export['followup'] = dict(base_commit=NEW_SHA,
    source_sha256=hashlib.sha256(source_path.read_bytes()).hexdigest(),
    patch_sha256=hashlib.sha256(patch.encode()).hexdigest(),
    measurements=FOLLOWUP_REPORTS,
    pooled_previous_batch1_fps=240/statistics.median(previous_times),
    pooled_followup_batch1_fps=240/statistics.median(followup_times),
    parity=FOLLOWUP_PARITY)
benchmark_export['limitations'] = ['Controlled generated clip, not the original user video.',
    'Batch-1 measurements vary between processes; do not claim a universal speedup.',
    'Raw stride-3 may begin at a different frame; only compare the Klygo revisions for parity.',
    'Initial strict hash differences were investigated with numerical comparisons.',
    'Follow-up is identified by base commit plus exact source and patch hashes.']
print(json.dumps(benchmark_export, indent=2))


{
  "gpu": "Tesla T4",
  "torch": "2.14.1+cu130",
  "ultralytics": "8.4.208",
  "opencv": "4.14.0",
  "baseline": "fb0f42f4075cb2bfab250446c1df059a67b5b2bb",
  "optimized": "9ff1abd3f1b39f594144d9a81373e7109998bf86",
  "video_frames": 240,
  "video_sha256": "a1f8b78d4bd9d9a1cf71ab63db4e2b4bbdba42523dbdce1548d919f9f6beba78",
  "weights_sha256": "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36",
  "primary": [
    {
      "name": "raw_b1",
      "revision": "raw",
      "kind": "raw",
      "batch": 1,
      "workers": 1,
      "prefetch": 1,
      "stride": 1,
      "fps": 85.85796291317266,
      "seconds": [
        2.795314399000006,
        2.641932875000009,
        3.5610583150000252
      ],
      "frames": [
        240,
        240,
        240
      ],
      "objects": [
        1321,
        1321,
        1321
      ],
      "signatures": [
        "7d491939ffaacc427ac99ec163758c6b584d62770f4a79a6f10b027797259b45",
        "7d491939ffaacc427ac99ec163758c6b584

# Final T4 follow-up

YOLOv8n, FP32, imgsz 640, controlled 240-frame 1280x720 clip.

| Configuration | Previous optimization | Final follow-up |
| --- | ---: | ---: |
| Managed batch 1, worker 1 (pooled 6 timings) | 75.55 FPS | 77.34 FPS |
| Managed batch 4, workers 4 | 122.30 FPS | 128.67 FPS |

Batch 1 improved about 2.4% in this run; variation prevents a universal speedup claim. Batch 4 values were measured in separate phases. All 25 case reports, individual timings, numerical parity checks, source hashes and reproduction code are retained.

Public API unchanged. Final source SHA256: `5ac4d0bc083821a384d9517a6e2438d8f9ed4a7bee97c634905e29954589241e`.

In [12]:
# Six-mode suite: install and verify the current source in a fresh checkout.
import os, sys, subprocess
from pathlib import Path
SUITE = Path('/content/klygo-six-mode')
PIN = '71f1055ed2a588efacbda8de28a89f244fad50d1'
if not SUITE.exists():
    subprocess.run(['git','clone','--quiet','--branch','perf/managed-input-pipeline','--single-branch','https://github.com/IchigoMazone/klygo.git',str(SUITE)],check=True)
subprocess.run(['git','-C',str(SUITE),'checkout','--quiet','--detach',PIN],check=True)
print('Source:', subprocess.check_output(['git','-C',str(SUITE),'rev-parse','HEAD'],text=True).strip(),flush=True)
print('Installing model extras; preserving the existing CUDA Torch build...',flush=True)
r = subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(SUITE),'ultralytics==8.4.208','transformers==4.57.1','accelerate','keras-hub','timm','open_clip_torch','nvidia-ml-py'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(r.stdout[-3500:])
assert r.returncode == 0
r = subprocess.run([sys.executable,'-c',"import torch,transformers;print('Torch',torch.__version__,'Transformers',transformers.__version__);assert torch.cuda.is_available();print('GPU',torch.cuda.get_device_name(0));print('VRAM GiB',torch.cuda.get_device_properties(0).total_memory/2**30)"],capture_output=True,text=True)
print(r.stdout);print(r.stderr[-1000:]);assert r.returncode==0


Source: 71f1055ed2a588efacbda8de28a89f244fad50d1
Installing model extras; preserving the existing CUDA Torch build...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.

Torch 2.14.1+cu130 Transformers 4.57.1
GPU Tesla T4
VRAM GiB 14.56317138671875




In [13]:
# Run the published benchmark code in isolated GPU processes, one job at a time.
RUN_PIN = '313897b4b40c8ebfd366ac1d99460727ed48919e'
subprocess.run(['git','-C',str(SUITE),'fetch','--quiet','origin','perf/managed-input-pipeline'],check=True)
subprocess.run(['git','-C',str(SUITE),'checkout','--quiet','--detach',RUN_PIN],check=True)
os.environ['PYTHONPATH']=str(SUITE)
sys.path.insert(0,str(SUITE))
from klygo import files
RUN_ROOT = SUITE/'artifacts/benchmarks/colab_t4_six_modes'
files.mkdir(RUN_ROOT)
print('Pinned source',RUN_PIN,flush=True)
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,driver_version,memory.total,power.limit','--format=csv,noheader'],text=True),flush=True)
MODEL_KEYS=['yolo','yolo11','yolo26','yolo-world','grounding-dino','grounding-dino-base','faster-rcnn','d-fine','locate-anything']
JOBS=[(key,'images',1) for key in MODEL_KEYS]+[(key,'images',4) for key in MODEL_KEYS if key!='locate-anything']+[(key,'video',batch) for key in MODEL_KEYS[:4] for batch in (1,4)]
JOB_RECORDS=[]
for index,(key,profile,batch) in enumerate(JOBS):
    folder=RUN_ROOT/(profile+'_b'+str(batch))
    files.mkdir(folder)
    destination=folder/(key+'.json')
    print(f'[{index+1}/{len(JOBS)}] {key} {profile} batch={batch}',flush=True)
    args=[sys.executable,'-u',str(SUITE/'benchmarks/gpu_suite/run.py'),'--model',key,'--device','cuda:0','--imgsz','640','--batch',str(batch),'--iterations','1' if key=='locate-anything' else '8','--repeats','7','--profile',profile,'--video-frames','120','--output',str(folder)]
    if key=='locate-anything': args += ['--allow-remote-code']
    process=subprocess.Popen(args,cwd=SUITE,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    lines=[]
    for line in process.stdout:
        lines.append(line)
        if line.startswith(('Loaded ','RESULT ','Traceback','ValueError:','RuntimeError:','ImportError:','PermissionError:')) or ' repeat ' in line: print(line.rstrip(),flush=True)
    returncode=process.wait()
    files.save(folder/(key+'.log'),''.join(lines),overwrite=True,verbose=False)
    record=dict(model=key,profile=profile,batch=batch,returncode=returncode,report=str(destination))
    JOB_RECORDS.append(record)
    files.save(RUN_ROOT/'jobs.json',JOB_RECORDS,overwrite=True,verbose=False)
    if returncode: print('FAILED: '+''.join(lines)[-4500:],flush=True)
print('GPU MATRIX FINISHED',flush=True)
print(JOB_RECORDS,flush=True)


Pinned source 313897b4b40c8ebfd366ac1d99460727ed48919e
Tesla T4, 580.82.07, 15360 MiB, 70.00 W

[1/25] yolo images batch=1
Loaded yolo: 0.51s, dtype=torch.float32
yolo repeat 1/7: raw-non-stream,raw-stream,native-non-stream,native-stream,managed-non-stream,managed-stream
yolo repeat 2/7: managed-stream,managed-non-stream,native-stream,native-non-stream,raw-stream,raw-non-stream
yolo repeat 3/7: raw-non-stream,raw-stream,native-non-stream,native-stream,managed-non-stream,managed-stream
yolo repeat 4/7: managed-stream,managed-non-stream,native-stream,native-non-stream,raw-stream,raw-non-stream
yolo repeat 5/7: raw-non-stream,raw-stream,native-non-stream,native-stream,managed-non-stream,managed-stream
yolo repeat 6/7: managed-stream,managed-non-stream,native-stream,native-non-stream,raw-stream,raw-non-stream
yolo repeat 7/7: raw-non-stream,raw-stream,native-non-stream,native-stream,managed-non-stream,managed-stream
RESULT yolo raw-non-stream: 99.568 FPS; parity=True
RESULT yolo raw-stream

In [14]:
# Retry the pinned LocateAnything model after installing its declared extras.
print('Installing LocateAnything processor dependencies...',flush=True)
r=subprocess.run([sys.executable,'-m','pip','install','-q','decord','lmdb'],capture_output=True,text=True)
print(r.stdout[-1000:]);print(r.stderr[-1800:]);assert r.returncode==0
REPAIR_ROOT=RUN_ROOT/'repair_locate'
files.mkdir(REPAIR_ROOT)
args=[sys.executable,'-u',str(SUITE/'benchmarks/gpu_suite/run.py'),'--model','locate-anything','--device','cuda:0','--imgsz','640','--batch','1','--iterations','1','--repeats','7','--allow-remote-code','--output',str(REPAIR_ROOT)]
print('LOCATE RETRY START',flush=True)
p=subprocess.Popen(args,cwd=SUITE,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
repair_lines=[]
for line in p.stdout:
    repair_lines.append(line)
    if line.startswith(('Loaded ','RESULT ','Traceback','ValueError:','RuntimeError:','ImportError:')) or ' repeat ' in line: print(line.rstrip(),flush=True)
REPAIR_STATUS=p.wait()
files.save(REPAIR_ROOT/'locate-anything.log',''.join(repair_lines),overwrite=True,verbose=False)
print('LOCATE RETRY RETURN CODE',REPAIR_STATUS,flush=True)
if REPAIR_STATUS: print(''.join(repair_lines)[-6500:])


Installing LocateAnything processor dependencies...


LOCATE RETRY START
Loaded locate-anything: 92.84s, dtype=torch.float16
locate-anything repeat 1/7: raw-non-stream,native-non-stream,managed-non-stream,managed-stream
locate-anything repeat 2/7: managed-stream,managed-non-stream,native-non-stream,raw-non-stream
locate-anything repeat 3/7: raw-non-stream,native-non-stream,managed-non-stream,managed-stream
locate-anything repeat 4/7: managed-stream,managed-non-stream,native-non-stream,raw-non-stream
locate-anything repeat 5/7: raw-non-stream,native-non-stream,managed-non-stream,managed-stream
locate-anything repeat 6/7: managed-stream,managed-non-stream,native-non-stream,raw-non-stream
locate-anything repeat 7/7: raw-non-stream,native-non-stream,managed-non-stream,managed-stream
RESULT locate-anything raw-non-stream: 1.561 FPS; parity=True
RESULT locate-anything native-non-stream: 1.546 FPS; parity=True
RESULT locate-anything managed-non-stream: 1.550 FPS; parity=True
RESULT locate-any

In [15]:
# Correct only the Ultralytics rows affected by persistent model_stream state.
# Earlier affected measurements remain archived and are excluded from final tables.
FIX_PIN='0750650314652a850503fc1b7d2a80a801d20291'
subprocess.run(['git','-C',str(SUITE),'fetch','--quiet','origin','perf/managed-input-pipeline'],check=True)
subprocess.run(['git','-C',str(SUITE),'checkout','--quiet','--detach',FIX_PIN],check=True)
CORRECTED_ROOT=RUN_ROOT/'corrected'
CORRECTED_RECORDS=[]
CORRECTED_JOBS=[(key,profile,batch) for key in MODEL_KEYS[:4] for profile in ('images','video') for batch in (1,4)]
for index,(key,profile,batch) in enumerate(CORRECTED_JOBS):
    folder=CORRECTED_ROOT/(profile+'_b'+str(batch));files.mkdir(folder)
    print(f'CORRECTED [{index+1}/{len(CORRECTED_JOBS)}] {key} {profile} batch={batch}',flush=True)
    args=[sys.executable,'-u',str(SUITE/'benchmarks/gpu_suite/run.py'),'--model',key,'--device','cuda:0','--imgsz','640','--batch',str(batch),'--iterations','8','--repeats','7','--profile',profile,'--video-frames','120','--output',str(folder)]
    p=subprocess.Popen(args,cwd=SUITE,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    lines=[]
    for line in p.stdout:
        lines.append(line)
        if line.startswith(('Loaded ','RESULT ','Traceback','ValueError:','RuntimeError:','ImportError:')) or ' repeat ' in line: print(line.rstrip(),flush=True)
    rc=p.wait();files.save(folder/(key+'.log'),''.join(lines),overwrite=True,verbose=False)
    CORRECTED_RECORDS.append(dict(model=key,profile=profile,batch=batch,returncode=rc,report=str(folder/(key+'.json'))))
    files.save(CORRECTED_ROOT/'jobs.json',CORRECTED_RECORDS,overwrite=True,verbose=False)
    if rc: print(''.join(lines)[-4500:],flush=True)
print('CORRECTED GPU MATRIX FINISHED',flush=True)


CORRECTED [1/16] yolo images batch=1
Loaded yolo: 0.38s, dtype=torch.float32
Traceback (most recent call last):
ValueError: Parameter group 'model' is locked at predict time. Runtime-overridable groups: ('post',).

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in th

In [16]:
# Verify actual frozen-group routing and the final benchmark adapter on the GPU.
FINAL_CODE_PIN='5f9aef9a0b7ed1fa93b51425bb232ad7d7de64e4'
subprocess.run(['git','-C',str(SUITE),'fetch','--quiet','origin','perf/managed-input-pipeline'],check=True)
subprocess.run(['git','-C',str(SUITE),'checkout','--quiet','--detach',FINAL_CODE_PIN],check=True)
assert subprocess.run(['git','-C',str(SUITE),'diff','--quiet',RUN_PIN,FINAL_CODE_PIN,'--','klygo','pyproject.toml']).returncode==0
r=subprocess.run([sys.executable,'-m','unittest','test.models.test_gpu_benchmark_modes','-v'],cwd=SUITE,env=os.environ.copy(),capture_output=True,text=True)
print(r.stdout);print(r.stderr);assert r.returncode==0
VERIFY_ROOT=RUN_ROOT/'control_verification';files.mkdir(VERIFY_ROOT)
p=subprocess.Popen([sys.executable,'-u',str(SUITE/'benchmarks/gpu_suite/run.py'),'--model','yolo','--device','cuda:0','--imgsz','640','--batch','1','--iterations','8','--repeats','7','--output',str(VERIFY_ROOT)],cwd=SUITE,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
verify_lines=[]
for line in p.stdout:
    verify_lines.append(line)
    if line.startswith(('Loaded ','RESULT ','Traceback')) or ' repeat ' in line: print(line.rstrip(),flush=True)
VERIFY_RC=p.wait();files.save(VERIFY_ROOT/'yolo.log',''.join(verify_lines),overwrite=True,verbose=False)
print('CONTROL VERIFICATION RETURN CODE',VERIFY_RC,flush=True)
if VERIFY_RC: print(''.join(verify_lines)[-5000:])



test_frozen_model_group_does_not_remember_native_stream_override (test.models.test_gpu_benchmark_modes.TestGpuBenchmarkModes.test_frozen_model_group_does_not_remember_native_stream_override) ... ok
test_modes_use_transient_native_override_and_frozen_managed_default (test.models.test_gpu_benchmark_modes.TestGpuBenchmarkModes.test_modes_use_transient_native_override_and_frozen_managed_default) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.010s

OK

Loaded yolo: 0.35s, dtype=torch.float32
yolo repeat 1/7: raw-non-stream,raw-stream,native-non-stream,native-stream,managed-non-stream,managed-stream
yolo repeat 2/7: managed-stream,managed-non-stream,native-stream,native-non-stream,raw-stream,raw-non-stream
yolo repeat 3/7: raw-non-stream,raw-stream,native-non-stream,native-stream,managed-non-stream,managed-stream
yolo repeat 4/7: managed-stream,managed-non-stream,native-stream,native-non-stream,raw-stream,raw-non-stream
yolo repeat 5/7: raw-n

In [18]:
bundle={'reports':selected,'jobs':JOB_RECORDS,'failed_attempts':[files.load(RUN_ROOT/'images_b1/locate-anything.json',verbose=False)],'rejected_control_probe_attempts':[files.load(r['report'],verbose=False) for r in CORRECTED_RECORDS],'control_verification':files.load(VERIFY_ROOT/'yolo.json',verbose=False),'hardware_details':{'nvidia_smi':subprocess.check_output(['nvidia-smi','--query-gpu=name,driver_version,memory.total,power.limit','--format=csv,noheader'],text=True).strip(),'cpu':subprocess.check_output(['lscpu'],text=True).split('Model name:')[-1].splitlines()[0].strip(),'core_identical_313_to_5f9':True}}
ok=[c for r in selected for c in r['cases'] if c['status']=='ok']
print('SELECTED',len(selected),'MEASURED',len(ok),'REPEATS',sum(len(c['runs']) for c in ok),'PARITY',all(c['all_parity_pass'] for c in ok))
files.save(RUN_ROOT/'bundle.json',bundle,overwrite=True,verbose=False)
export_text=json.dumps(bundle,separators=(',',':'),ensure_ascii=True)
display(HTML('<textarea readonly aria-label="Benchmark JSON export" rows="5" style="width:100%">'+html.escape(export_text)+'</textarea>'))
print('EXPORT_READY',len(export_text),hashlib.sha256(export_text.encode()).hexdigest())


Output hidden; open in https://colab.research.google.com to view.

In [19]:
# Sustained managed stream: 600 seconds, no retained frames.
p=subprocess.Popen([sys.executable,'-u',str(SUITE/'benchmarks/gpu_suite/stability.py'),str(RUN_ROOT/'video_b1/bus_motion_h264.mp4'),'--seconds','600','--output',str(RUN_ROOT/'stability.json')],cwd=SUITE,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
stability_lines=[]
for line in p.stdout:
    stability_lines.append(line)
    if line.startswith(('STABILITY','Traceback','{')): print(line.rstrip(),flush=True)
STABILITY_RC=p.wait()
files.save(RUN_ROOT/'stability.log',''.join(stability_lines),overwrite=True,verbose=False)
print('STABILITY_RETURN_CODE',STABILITY_RC,flush=True)
if STABILITY_RC: print(''.join(stability_lines)[-6000:])


STABILITY 39s: 72.99 FPS; RSS=1618.9 MiB; fds=45
STABILITY 78s: 73.27 FPS; RSS=1608.4 MiB; fds=45
STABILITY 116s: 68.47 FPS; RSS=1609.1 MiB; fds=45
STABILITY 155s: 56.23 FPS; RSS=1608.5 MiB; fds=45
STABILITY 194s: 49.62 FPS; RSS=1608.5 MiB; fds=45
STABILITY 232s: 49.36 FPS; RSS=1608.5 MiB; fds=45
STABILITY 271s: 52.88 FPS; RSS=1608.5 MiB; fds=45
STABILITY 309s: 63.08 FPS; RSS=1608.6 MiB; fds=45
STABILITY 348s: 77.43 FPS; RSS=1616.5 MiB; fds=45
STABILITY 386s: 76.19 FPS; RSS=1616.5 MiB; fds=45
STABILITY 425s: 74.01 FPS; RSS=1608.7 MiB; fds=45
STABILITY 464s: 74.54 FPS; RSS=1619.2 MiB; fds=45
STABILITY 502s: 77.57 FPS; RSS=1616.6 MiB; fds=45
STABILITY 540s: 72.73 FPS; RSS=1616.6 MiB; fds=45
STABILITY 579s: 73.64 FPS; RSS=1608.7 MiB; fds=45
STABILITY FINISHED True 0.019429218143973692
STABILITY_RETURN_CODE 0


In [21]:
from IPython.display import Markdown
stability=files.load(RUN_ROOT/'stability.json',verbose=False)
bundle['stability']=stability
files.save(RUN_ROOT/'bundle.json',bundle,overwrite=True,verbose=False)
stability_text=json.dumps(stability,separators=(',',':'),ensure_ascii=True)
display(HTML('<textarea readonly aria-label="Stability JSON export" rows="3" style="width:100%">'+html.escape(stability_text)+'</textarea>'))
print('STABILITY_EXPORT',len(stability_text),hashlib.sha256(stability_text.encode()).hexdigest())
modes=['raw-non-stream','raw-stream','native-non-stream','native-stream','managed-non-stream','managed-stream']
summary=['## Final Tesla T4 GPU results','','25 configurations / 124 supported cases / 868 passes / all selected parity PASS.','','### H.264 video, batch 1: median FPS','','| Model | Raw NS | Raw S | Native NS | Native S | Managed NS | Managed S |','|---|---:|---:|---:|---:|---:|---:|']
for report in selected:
    if report['settings']['profile']=='video' and report['settings']['batch']==1:
        cases={c['mode']:c for c in report['cases']}
        summary.append('|'+report['model']+'|'+'|'.join(f"{cases[m]['fps_median']:.2f}" for m in modes)+'|')
summary += ['','### 600-second non-retaining managed stream','',f"Completed {stability['elapsed_seconds']:.2f} s; {stability['total_frames']} ordered frames; exceptions {len(stability['exceptions'])}; memory/FD gates PASS={stability['pass']}.",'','| Metric | First minute median | Last minute median | Change % |','|---|---:|---:|---:|']
for key in ['rss_mib','vram_allocated_mib','fps','open_fds']:
    summary.append(f"|{key}|{stability[key+'_first_window']:.2f}|{stability[key+'_last_window']:.2f}|{stability[key+'_growth_percent']:.2f}|")
summary+=['','FPS fluctuations remain in raw samples. Passing memory/FD gates is not a production throughput guarantee. Unsupported upstream modes are N/A in the full nine-model matrices.']
display(Markdown(chr(10).join(summary)))


STABILITY_EXPORT 63945 77b72b61d13918946b10199806bbadc9672db8305d940e4fceefd5beb3514e60


## Final Tesla T4 GPU results

25 configurations / 124 supported cases / 868 passes / all selected parity PASS.

### H.264 video, batch 1: median FPS

| Model | Raw NS | Raw S | Native NS | Native S | Managed NS | Managed S |
|---|---:|---:|---:|---:|---:|---:|
|yolo|82.13|84.03|74.41|71.23|70.51|70.03|
|yolo11|70.50|63.47|62.02|62.79|59.04|59.38|
|yolo26|62.95|63.75|60.86|55.88|54.97|56.03|
|yolo-world|64.26|64.67|62.20|58.48|58.05|56.80|

### 600-second non-retaining managed stream

Completed 601.77 s; 37440 ordered frames; exceptions 0; memory/FD gates PASS=True.

| Metric | First minute median | Last minute median | Change % |
|---|---:|---:|---:|
|rss_mib|1608.40|1608.71|0.02|
|vram_allocated_mib|12.10|12.10|0.00|
|fps|72.45|74.72|3.13|
|open_fds|45.00|45.00|0.00|

FPS fluctuations remain in raw samples. Passing memory/FD gates is not a production throughput guarantee. Unsupported upstream modes are N/A in the full nine-model matrices.